In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.naumov.lesson3 import Exercise

In [ ]:
digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target

print(f"Признаков: {X.shape[1]}, Классов: {len(np.unique(y))}")

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=123)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=123)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print(f"После нормировки: mean={X_train.mean():.3f}, std={X_train.std():.3f}")

In [ ]:
def evaluate(lr, batch_size, hidden_size, seed):
    rng = np.random.default_rng(seed)

    model = Exercise.create_model(
        Exercise.create_linear_layer(64, hidden_size, rng),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(hidden_size, 10, rng),
        Exercise.create_logsoftmax_layer(),
    )
    loss = Exercise.create_cross_entropy_loss()

    Exercise.train_model(model, loss, X_train, y_train, lr=lr, n_epoch=30, batch_size=batch_size)

    val_out = model.forward(X_val)
    val_acc = np.mean(np.argmax(val_out, axis=1) == y_val)
    return val_acc

In [ ]:
print("Подбор гиперпараметров")

lr_list = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
batch_list = [16, 32, 64, 128]
hidden_list = [16, 32, 64, 96, 128]
seeds = [7, 42, 123]

best_acc = 0
best_params = {}

total = len(lr_list) * len(batch_list) * len(hidden_list)
count = 0

for lr in lr_list:
    for batch in batch_list:
        for hidden in hidden_list:
            count += 1

            accs = []
            for seed in seeds:
                acc = evaluate(lr, batch, hidden, seed)
                accs.append(acc)
            mean_acc = np.mean(accs)

            if mean_acc > best_acc:
                best_acc = mean_acc
                best_params = {"lr": lr, "batch": batch, "hidden": hidden}
                print(f"[{count}/{total}] NEW BEST: {mean_acc:.4f} | lr={lr}, batch={batch}, hidden={hidden}")
            elif count % 10 == 0:
                print(f"[{count}/{total}] {mean_acc:.4f}")

print(f"\nЛучшие параметры: lr={best_params['lr']}, batch={best_params['batch']}, hidden={best_params['hidden']}")
print(f"Val accuracy: {best_acc:.4f}")

In [ ]:
print("Финальное обучение")

X_full = np.vstack([X_train, X_val])
y_full = np.hstack([y_train, y_val])

test_accs = []

for seed in [7, 42, 123]:
    print(f"Seed {seed}:")
    rng = np.random.default_rng(seed)

    hidden_size = int(best_params["hidden"])

    model = Exercise.create_model(
        Exercise.create_linear_layer(64, hidden_size, rng),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(hidden_size, 10, rng),
        Exercise.create_logsoftmax_layer(),
    )
    loss = Exercise.create_cross_entropy_loss()

    batch_size = int(best_params["batch"])

    Exercise.train_model(model, loss, X_full, y_full, lr=best_params["lr"], n_epoch=30, batch_size=batch_size)

    test_out = model.forward(X_test)
    test_acc = np.mean(np.argmax(test_out, axis=1) == y_test)
    test_accs.append(test_acc)
    print(f"  Test: {test_acc:.4f}")

print(f"\nИтог: {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")